# RAG For Strategy Team


## Setup

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # prevents Apple Silicon crash
os.environ.setdefault("OMP_NUM_THREADS", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["HF_HOME"] = str(ROOT / "hf_cache")           # models/datasets downloaded once, stored here
DATA_DIR, CACHE_DIR, OUTPUTS_DIR, EVAL_DIR = (ROOT / d for d in ("data", "cache", "outputs", "eval"))
for d in (DATA_DIR, CACHE_DIR, OUTPUTS_DIR, EVAL_DIR): d.mkdir(exist_ok=True)
from dotenv import load_dotenv; load_dotenv(ROOT / ".env")  # OPENAI_API_KEY

import re
import json
import random
import time as time_module

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = random.Random(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

CONFIG = {
    "seed": SEED,
    "device": DEVICE,
    "emb_model": "sentence-transformers/all-MiniLM-L6-v2",
    "emb_batch_size": 64,
    "chunk_words": 150,
    "chunk_overlap": 30,
    "retrieve_pool": 20,
    "top_k": 5,
    "llm_model": os.environ.get("LLM_MODEL", "gpt-4o-mini"),
    "answer_max_words": 150,
    "corpus_sample_size": 15000,
}

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
HAS_LLM_KEY = bool(OPENAI_API_KEY) and OPENAI_API_KEY != "REPLACE_ME"
print(f"LLM key configured: {HAS_LLM_KEY}")
CONFIG

Using device: mps
LLM key configured: True


{'seed': 42,
 'device': 'mps',
 'emb_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'emb_batch_size': 64,
 'chunk_words': 150,
 'chunk_overlap': 30,
 'retrieve_pool': 20,
 'top_k': 5,
 'llm_model': 'gpt-4o-mini',
 'answer_max_words': 150,
 'corpus_sample_size': 15000}

## 7.1 Ingestion

Corpus = internal documents classified as "tech" by notebook 02's group classifier
(`cache/cnn_hl_emb.npy` + `cache/group_emb.npy`, run over all 306,913 CNN/DailyMail rows). Not just
the 4,562 rows that also matched one of the 32 AG-News topics (`data/cnn_matched.parquet`); the
wider tech pool (~50k rows) gives fact/story questions a much better chance their article actually
exists in the search index.

(The gap/strategy table in section 7.5 is built from notebook 02's own `outputs/gap_ranking.csv`,
not from `cnn_matched.parquet` directly - see that section for why.)

In [2]:
# Reuse notebook 02's classifier to tag each of the 306,913 articles as tech/business/science/general.
# Compare each headline's embedding to the 4 group embeddings and keep whichever one is closest.
GROUP_NAMES = ["tech", "business", "science", "general"]
group_emb = np.load(CACHE_DIR / "group_emb.npy")
cnn_hl_emb = np.load(CACHE_DIR / "cnn_hl_emb.npy")
cnn_hl_ids = np.load(CACHE_DIR / "cnn_hl_ids.npy", allow_pickle=True)

cnn_group = (cnn_hl_emb @ group_emb.T).argmax(axis=1)
tech_ids = cnn_hl_ids[cnn_group == GROUP_NAMES.index("tech")]
print(f"Tech-classified internal documents available: {len(tech_ids):,}")

# Fixed seed, so the sample is the same 15,000 articles on every rerun.
sample_size = min(CONFIG["corpus_sample_size"], len(tech_ids))
sample_ids = np.random.RandomState(SEED).choice(tech_ids, size=sample_size, replace=False)
corpus_ids = set(sample_ids)
print(f"Sampled RAG corpus: {len(corpus_ids):,} tech articles")

Tech-classified internal documents available: 50,873
Sampled RAG corpus: 15,000 tech articles


In [3]:
# Pull full article text for just the sampled ids. Filtering the Arrow dataset like this avoids
# loading all 306,913 articles into memory at once.
from datasets import Dataset as HFDataset

cnn_full = HFDataset.from_parquet(str(DATA_DIR / "cnn_clean.parquet"))
cnn_full = cnn_full.filter(lambda ex: ex["id"] in corpus_ids, desc="Selecting RAG corpus documents")
print(f"Selected {len(cnn_full):,} full articles")

corpus_df = cnn_full.to_pandas()
corpus_df.head(3)

Selected 15,000 full articles


,id,article,highlights
0,77d7c8cf2a9432e395d629371a12790c563c19f7,"BREMEN, Germany -- Carlos Alberto, who scored ...",Werder Bremen pay a club record $10.7 million ...
1,05688bac0e46542be65a6df240428a6f26c86fd0,"JOHANNESBURG, South Africa -- South African fa...",South Africa lead New Zealand by 287 with 8 wi...
2,a228e744966fe03eca515b971329acacf9feabb2,"GLASGOW, Scotland -- World champions Italy cli...",Italy win 2-1 in Scotland to secure their qual...


`doc_label` = first highlight sentence, truncated to 100 characters - CNN/DailyMail has no
titles, so this is the citation label.

In [4]:
def first_highlight_sentence(highlights, max_len=100):
    parts = highlights.split(" . ")
    first = parts[0].strip() if parts and parts[0].strip() else highlights.strip()
    if len(first) > max_len:
        first = first[:max_len].rsplit(" ", 1)[0] + "\u2026"
    return first

corpus_df["doc_label"] = corpus_df["highlights"].map(first_highlight_sentence)
corpus_df[["id", "doc_label"]].head(5)

,id,doc_label
0,77d7c8cf2a9432e395d629371a12790c563c19f7,Werder Bremen pay a club record $10.7 million ...
1,05688bac0e46542be65a6df240428a6f26c86fd0,South Africa lead New Zealand by 287 with 8 wi...
2,a228e744966fe03eca515b971329acacf9feabb2,Italy win 2-1 in Scotland to secure their qual...
3,373ef5be1d84b8c4905193e569e0cf8a7ead3d03,Gloss taking backseat to bold colors in lipsticks
4,282f1910bd57e50d4401c695ebe75b817de68548,Brazilian Kaka is named European player of the...


In [5]:
# Split an article into overlapping 150-word pieces. The embedding model only reads about 256
# words at a time, so a full article would get cut off. The 30-word overlap means a fact sitting
# right on the boundary between two pieces still shows up whole in one of them.
def chunk_text(text, chunk_words, overlap_words):
    words = text.split()
    if not words:
        return []
    step = chunk_words - overlap_words
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_words]))
        if start + chunk_words >= len(words):
            break
        start += step
    return chunks

chunks_path = DATA_DIR / "chunks.parquet"
chunks_df = pd.read_parquet(chunks_path) if chunks_path.exists() else None
if chunks_df is not None and set(chunks_df["doc_id"]) != corpus_ids:
    print("Cached chunks belong to a different corpus; rebuilding")
    chunks_df = None

if chunks_df is not None:
    print(f"Loaded cached chunks: {len(chunks_df):,}")
else:
    # Chunk every article once and save the result. Re-running this notebook later just loads the
    # file above instead of redoing this loop.
    records = []
    for row in tqdm(corpus_df.itertuples(index=False), total=len(corpus_df), desc="Chunking"):
        for idx, piece in enumerate(chunk_text(row.article, CONFIG["chunk_words"], CONFIG["chunk_overlap"])):
            records.append({
                "chunk_id": f"{row.id}_{idx}", "doc_id": row.id, "chunk_idx": idx,
                "text": piece, "doc_label": row.doc_label,
            })
    chunks_df = pd.DataFrame(records)
    chunks_df.to_parquet(chunks_path, index=False)
    print(f"Created {len(chunks_df):,} chunks from {len(corpus_df):,} documents")

chunks_df.head(3)

Loaded cached chunks: 86,334


,chunk_id,doc_id,chunk_idx,text,doc_label
0,77d7c8cf2a9432e395d629371a12790c563c19f7_0,77d7c8cf2a9432e395d629371a12790c563c19f7,0,"BREMEN, Germany -- Carlos Alberto, who scored ...",Werder Bremen pay a club record $10.7 million ...
1,77d7c8cf2a9432e395d629371a12790c563c19f7_1,77d7c8cf2a9432e395d629371a12790c563c19f7,1,"Early in 2005, he moved to Corinthians, where ...",Werder Bremen pay a club record $10.7 million ...
2,77d7c8cf2a9432e395d629371a12790c563c19f7_2,77d7c8cf2a9432e395d629371a12790c563c19f7,2,played in the Bundesliga for Borussia Dortmund...,Werder Bremen pay a club record $10.7 million ...


## 7.2 Vector search

In [6]:
# Turn each chunk's text into a vector that captures its meaning. Two chunks about the same thing
# land close together even with different wording - that's what makes searching by meaning
# possible, not just keyword matching.
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(CONFIG["emb_model"], device=DEVICE)

chunk_emb_path = CACHE_DIR / "chunk_emb.npy"
chunk_embeddings = np.load(chunk_emb_path) if chunk_emb_path.exists() else None
if chunk_embeddings is not None and len(chunk_embeddings) != len(chunks_df):
    print("Cached chunk embeddings size mismatch; rebuilding")
    chunk_embeddings = None

if chunk_embeddings is not None:
    print(f"Loaded cached chunk embeddings: {chunk_embeddings.shape}")
else:
    # Slow step (minutes, not seconds). Save the result so it only has to run once.
    t0 = time_module.time()
    chunk_embeddings = embed_model.encode(
        chunks_df["text"].tolist(), batch_size=CONFIG["emb_batch_size"],
        show_progress_bar=True, normalize_embeddings=True,
    ).astype(np.float32)
    print(f"Embedded {len(chunks_df):,} chunks in {time_module.time() - t0:.1f}s")
    np.save(chunk_emb_path, chunk_embeddings)
chunk_embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded cached chunk embeddings: (86334, 384)


(86334, 384)

In [7]:
# FAISS finds the closest vectors out of thousands, fast. Build the index once here, from the
# vectors above.
import faiss

index_path = CACHE_DIR / "faiss.index"
index = None
if index_path.exists():
    index = faiss.read_index(str(index_path))
    if index.ntotal != len(chunks_df):
        print("Cached index size mismatch; rebuilding")
        index = None

if index is None:
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings.astype(np.float32))
    faiss.write_index(index, str(index_path))
    print(f"Built FAISS index with {index.ntotal:,} vectors")
else:
    print(f"Loaded cached FAISS index with {index.ntotal:,} vectors")

Loaded cached FAISS index with 86,334 vectors


In [8]:
# Find the top 5 relevant chunks for a question. Pull a wider pool (20) from FAISS first, then
# keep only the first chunk we see per article going down that list. Stops one long, relevant
# article from filling all 5 slots and crowding out others.
def retrieve(query, k=None):
    k = CONFIG["top_k"] if k is None else k
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_emb, CONFIG["retrieve_pool"])
    scores, idxs = scores[0], idxs[0]

    seen_docs = set()
    results = []
    for score, idx in zip(scores, idxs):
        if idx == -1:
            continue
        row = chunks_df.iloc[idx]
        if row.doc_id in seen_docs:
            continue
        seen_docs.add(row.doc_id)
        results.append({
            "chunk_id": row.chunk_id, "doc_id": row.doc_id, "doc_label": row.doc_label,
            "text": row.text, "score": float(score),
        })
        if len(results) >= k:
            break
    return results

# sanity check
sample_query = chunks_df.iloc[0]["text"][:80]
retrieve(sample_query, k=3)

[{'chunk_id': '77d7c8cf2a9432e395d629371a12790c563c19f7_0',
  'doc_id': '77d7c8cf2a9432e395d629371a12790c563c19f7',
  'doc_label': 'Werder Bremen pay a club record $10.7 million for Carlos Alberto',
  'text': 'BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão

## 7.3 Generation with citations

The model has two tools and picks the right one per question itself. No manual mode selection.

- `search_articles`: semantic search over article chunks, for fact/specific-story questions. Returns
  numbered snippets, the model cites them as `[n]`.
- `query_topics_db`: read-only SQL against the gap-data table built in §7.5, for gap/strategy
  questions. Structured counts, not text snippets, so these questions don't get `[n]` citations.

(Coverage-lookup questions - "how many articles do we have on X" for any topic - aren't supported
yet, that's a future enhancement, see 7.5.)

Any `[n]` the model cites that wasn't actually returned by `search_articles` gets flagged as invalid.
If `search_articles` finds nothing relevant, the model replies `Not found in the library.`

In [9]:
CITATION_RE = re.compile(r"\[(\d+)\]")

NOT_FOUND = "Not found in the library."

SYSTEM_PROMPT = (
    "You are a research assistant for a content strategy team. Answer using ONLY the two tools "
    "provided - never use outside knowledge. Use search_articles for questions about a specific fact, "
    "story or article. Use query_topics_db for gap/strategy questions comparing external demand to "
    "internal coverage across the 32 known topics. "
    "When you use search_articles, cite claims inline as [1], [2], etc. matching the returned source "
    "numbers exactly; do not invent numbers. When you use query_topics_db, state the numbers directly, "
    "no [n] citation needed. "
    f"If a tool call comes back with nothing relevant, do not call it again with a different query - "
    f"immediately reply exactly: {NOT_FOUND} "
    f"Keep the answer to {CONFIG['answer_max_words']} words or fewer."
)

# These two tool definitions get sent to OpenAI along with the question. The model reads the name
# and description and picks whichever one fits, if either does.
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_articles",
        "description": "Semantic search over internal news article chunks. Use for fact or "
                        "specific-story questions. Returns numbered snippets to cite as [n].",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "search query"}},
            "required": ["query"],
        },
    },
}

DB_TOOL = {
    "type": "function",
    "function": {
        "name": "query_topics_db",
        "description": (
            "Run a read-only SQL SELECT against `topic_gap_summary` (columns: topic_id, topic_name, "
            "ext_count, int_count, ext_share, int_share, coverage_ratio, coverage_level, gap, rank) "
            "- the 32 predefined topics, external vs internal coverage as a SHARE of each corpus (not "
            "raw counts, which aren't comparable since the external and internal corpora are "
            "different sizes). `gap` = ext_share - int_share: positive means under-covered relative "
            "to external demand, negative means over-covered. Always rank by `gap`, not by raw "
            "counts. Use for gap/strategy questions, e.g. "
            "SELECT topic_name, ext_share, int_share, gap, coverage_level FROM topic_gap_summary "
            "ORDER BY gap DESC."
        ),
        "parameters": {
            "type": "object",
            "properties": {"sql": {"type": "string", "description": "a single SELECT statement"}},
            "required": ["sql"],
        },
    },
}

TOOLS = [SEARCH_TOOL, DB_TOOL]

def ask(question, k=None):
    k = CONFIG["top_k"] if k is None else k
    all_sources = []  # every chunk search_articles has returned so far, numbered in order returned

    # Runs whichever tool the model asked for and returns the result to it.
    def tool_executor(name, arguments_json):
        args = json.loads(arguments_json)
        if name == "search_articles":
            retrieved = retrieve(args["query"], k=k)
            start_n = len(all_sources) + 1
            new_sources = [{"n": start_n + i, **r} for i, r in enumerate(retrieved)]
            all_sources.extend(new_sources)
            return json.dumps([{"n": s["n"], "text": s["text"]} for s in new_sources])
        elif name == "query_topics_db":
            sql = args["sql"].strip()
            if not sql.lower().startswith("select"):
                return json.dumps({"error": "only SELECT statements are allowed"})
            try:
                with sqlite3.connect(db_path) as conn:
                    result_df = pd.read_sql_query(sql, conn)
                return result_df.to_json(orient="records")
            except Exception as e:
                return json.dumps({"error": str(e)})
        return json.dumps({"error": f"unknown tool {name}"})

    try:
        answer = llm(SYSTEM_PROMPT, question, tools=TOOLS, tool_executor=tool_executor)
    except RuntimeError:
        # Model kept calling tools instead of settling on an answer (e.g. retrying a search
        # that found nothing) - fail safe instead of crashing the caller.
        answer = NOT_FOUND

    # Check the answer's [n] markers against what was actually retrieved, to catch any citation
    # number the model made up (shouldn't happen, but worth catching if it does).
    cited_ns = set(int(n) for n in CITATION_RE.findall(answer))
    valid_ns = set(s["n"] for s in all_sources)
    invalid_ns = sorted(cited_ns - valid_ns)

    citations = [
        {"n": s["n"], "doc_id": s["doc_id"], "doc_label": s["doc_label"],
         "chunk_id": s["chunk_id"], "snippet": s["text"][:200]}
        for s in all_sources if s["n"] in cited_ns
    ]
    return {"answer": answer, "citations": citations, "sources": all_sources, "retrieved": all_sources,
            "invalid_citation_markers": invalid_ns}

## 7.4 LLM wrapper

One function, `llm(system, user, tools=None, tool_executor=None) -> str`, using OpenAI (kept to a
single provider since the case study asks for a lightweight RAG solution). Supports tool/function
calling: if the model calls a tool, `tool_executor(name, arguments_json) -> str` runs it and feeds
the result back until the model returns a final answer (capped at a few rounds). Retries once per
API call on failure, raises immediately if no key is configured.

In [10]:
from openai import OpenAI

_openai_client = OpenAI(api_key=OPENAI_API_KEY) if HAS_LLM_KEY else None

def _call_openai(messages, tools, max_retries):
    last_err = None
    for attempt in range(max_retries + 1):
        try:
            return _openai_client.chat.completions.create(
                model=CONFIG["llm_model"], temperature=0,
                messages=messages, tools=tools if tools else None,
            )
        except Exception as e:
            last_err = e
            print(f"llm() call failed (attempt {attempt + 1}/{max_retries + 1}): {e!r}")
            if attempt < max_retries:
                time_module.sleep(2)
    raise last_err

# The model can't run tools itself, it can only ask for one and wait. So this loops: send the
# question, if the model asks for a tool run it and hand back the result, repeat until it answers
# in plain text instead of asking for another tool. Capped at a few rounds so a confused model
# can't keep asking forever.
def llm(system, user, tools=None, tool_executor=None, max_retries=1, max_tool_rounds=3):
    if _openai_client is None:
        raise RuntimeError("OPENAI_API_KEY is not configured in .env")

    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    for _ in range(max_tool_rounds + 1):
        resp = _call_openai(messages, tools, max_retries)
        msg = resp.choices[0].message
        tool_calls = msg.tool_calls
        if not tool_calls or tool_executor is None:
            return (msg.content or "").strip()

        messages.append({
            "role": "assistant", "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in tool_calls],
        })
        for tc in tool_calls:
            result = tool_executor(tc.function.name, tc.function.arguments)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    raise RuntimeError("Exceeded max tool-call rounds without a final answer")

## 7.5 Structured data (SQLite) - for gap/strategy questions

Plain vector search can't answer "what's our content gap on Y." It only returns similar-looking
text, it can't count. The gap data (32 AG-News-derived topics, external vs internal coverage as a
share of each corpus, already computed in notebook 02: `outputs/gap_ranking.csv`) lives instead in
a small SQLite DB (`cache/rag.db`), exposed to the LLM as a tool call. That keeps it out of the
chunk index, where it would risk getting blended with an unrelated article snippet.

We load `gap_ranking.csv` directly rather than recomputing counts here - raw external/internal
counts aren't comparable on their own (the external and internal corpora are very different sizes),
which is exactly why notebook 02 computed shares and a `gap` column in the first place. Reusing that
same file keeps this table consistent with notebook 02's own ranking.

**Scoped for this pass:** we're shipping the two question types with validated methodology behind
them from Parts 1–3 - fact/story retrieval (chunk search) and gap/strategy analysis (the same
external-vs-internal comparison from notebook 02). A third type, open-ended coverage-lookup ("how
many articles do we have on any topic, not just the 32 known ones"), would need its own
topic-extraction step tuned for this internal corpus rather than reusing notebook 01's external-
content settings. Scoping that as a follow-on enhancement rather than shipping a rough version now.

### 7.5.1 Build the SQLite DB

In [11]:
# Load notebook 02's own gap_ranking.csv (topic_id, topic_name, ext_count, int_count, ext_share,
# int_share, coverage_ratio, coverage_level, gap, rank) straight into SQLite - no recomputation,
# so this table always matches notebook 02's own share-based ranking.
import sqlite3

db_path = CACHE_DIR / "rag.db"

topic_gap_summary = pd.read_csv(OUTPUTS_DIR / "gap_ranking.csv")

with sqlite3.connect(db_path) as conn:
    topic_gap_summary.to_sql("topic_gap_summary", conn, if_exists="replace", index=False)

print(f"Saved {db_path}")
print(f"topic_gap_summary: {len(topic_gap_summary)} rows")
topic_gap_summary.sort_values("rank").head(5)

Saved /Users/shubhamyadav/Desktop/topic_modelling/case-study/cache/rag.db
topic_gap_summary: 32 rows


,topic_id,topic_name,ext_count,int_count,ext_share,int_share,coverage_ratio,coverage_level,gap,rank
0,7,Intel and AMD Processor Developments,1141,26,0.050605,0.005699,0.112621,Low,0.044906,1
1,6,Oracle Corp. Takeover Bid,882,1,0.039118,0.000219,0.005604,Low,0.038899,2
2,22,Oracle Enterprise Software Innovations,796,7,0.035304,0.001534,0.043463,Low,0.033770,3
3,26,Microsoft Windows Server Release 2006,1031,64,0.045727,0.014029,0.306800,Low,0.031698,4
4,12,Windows XP Security Updates,671,6,0.029760,0.001315,0.044194,Low,0.028445,5


## Helper for writing test questions

`find_docs(keyword)` lists corpus ids and highlights that mention a keyword, so you can pick a `gold_doc_id`.

In [12]:
def find_docs(keyword):
    mask = (corpus_df["highlights"].str.contains(keyword, case=False, regex=False)
            | corpus_df["article"].str.contains(keyword, case=False, regex=False))
    for r in corpus_df[mask].head(10).itertuples():
        print(r.id, "|", r.highlights)
        print()

## 7.6 Evaluation

Run `ask()` on every question in `eval/test_questions.csv`. The questions were written by hand, with columns `qid, question, type, gold_doc_id` (`type` is `answerable` or `unanswerable`; `gold_doc_id` is empty for unanswerable).

In [13]:
questions_path = EVAL_DIR / "test_questions.csv"
assert questions_path.exists(), "Missing eval/test_questions.csv (columns: qid, question, type, gold_doc_id)"
test_questions = pd.read_csv(questions_path, dtype=str, keep_default_na=False)
assert {"qid", "question", "type", "gold_doc_id"} <= set(test_questions.columns), "missing columns in test_questions.csv"
test_questions

,qid,question,type,gold_doc_id
0,q1,What reward did Microsoft offer for informatio...,answerable,5c5a8bc667c46f3cb85feb0535d7eeac5d5ad747
1,q2,What did the Chinese hackers claim to have acc...,answerable,d587d18b5bbe6edec12ba28fab028e8feacebcbf
2,q3,Which topics have the biggest gap between exte...,answerable,
3,q4,What software did developers build to let the ...,answerable,f0d93bdd383762936746b48f7e67ffe72cfb1ede
4,q5,What incident triggered the classified cyberat...,answerable,17662736c0b0056d18a33c078f04b64bc369ab0b
5,q6,What is cyclist Floyd Landis accused of hacking?,answerable,6215e369628ef593c7022b497ea4f219f40aad0a
6,q7,"What does the ""How old do I look"" website do?",answerable,4eed9e12497734b4420aebd1ca8351caa096ae1b
7,q8,"What does the ""Anatomy Lab"" iPhone app let use...",answerable,b69eedb05cf2556b1a450c6d017fcf1fff26ec97
8,q9,What did NASA announce about its 2024 Artemis ...,unanswerable,
9,q10,How much did OpenAI raise in its most recent f...,unanswerable,


In [14]:
# Ask every test question and save what comes back. hit_at_5 checks whether the right article
# showed up in the top 5 - only makes sense when there's a gold_doc_id. The gap question uses the
# SQL tool instead of search, so there's nothing to check it against; hit_at_5 stays blank for it.
all_asks = {}
result_rows = []

for row in test_questions.itertuples(index=False):
    result = ask(row.question)
    all_asks[row.qid] = result

    retrieved_ids = [r["doc_id"] for r in result["retrieved"]]
    has_gold = row.type == "answerable" and bool(row.gold_doc_id)
    hit_at_5 = int(row.gold_doc_id in retrieved_ids) if has_gold else ""

    print(f"=== {row.qid} [{row.type}] {row.question}")
    print("ANSWER   :", result["answer"])
    print("CITATIONS:", [(c["n"], c["doc_id"]) for c in result["citations"]])
    print("RETRIEVED:", retrieved_ids)
    print()

    result_rows.append({
        "qid": row.qid, "question": row.question, "type": row.type, "gold_doc_id": row.gold_doc_id,
        "answer": result["answer"],
        "citations": "; ".join(f"[{c['n']}] {c['doc_id']}" for c in result["citations"]),
        "retrieved_ids": "; ".join(retrieved_ids),
        "hit_at_5": hit_at_5,
        "citation_correct": "", "hallucination": "", "notes": "",
    })

rag_results = pd.DataFrame(result_rows)
rag_results.to_csv(OUTPUTS_DIR / "rag_results.csv", index=False)
print(f"Saved {OUTPUTS_DIR / 'rag_results.csv'} -> fill citation_correct / hallucination with Y/N, "
      "then save as eval/rag_results_labeled.csv")

=== q1 [answerable] What reward did Microsoft offer for information about the Conficker virus author?
ANSWER   : Microsoft offered a $250,000 reward for information leading to the identification of those behind the Conficker virus. This announcement was made in collaboration with other technology industry leaders as part of ongoing security efforts to protect customers from the threat posed by the virus, which had the potential to spread unchecked across networks [1].
CITATIONS: [(1, '5c5a8bc667c46f3cb85feb0535d7eeac5d5ad747')]
RETRIEVED: ['5c5a8bc667c46f3cb85feb0535d7eeac5d5ad747', '47a24a6c439ca794a338dc6ace74bafbb7c451b0', '09e0d49982c4d41904f37460a5cfbadb3bedf5c8', 'a9dd2d32a7be100ae87bd57b03070b4e9ce4d524', '04d60a388b8970bdc5db6c4c1ef628063bde643e']



=== q2 [answerable] What did the Chinese hackers claim to have accessed?
ANSWER   : Chinese hackers have been accused of attempting to access intelligence related to negotiations among the Group of 20 wealthy nations. Additionally, there were allegations that they stole blueprints for a new Australian spy headquarters, although these claims were dismissed by the Chinese government as groundless. The hackers have been linked to sophisticated intrusions targeting defense contractors, human rights groups, and major tech companies, typically seeking trade secrets, source code, or sensitive intelligence [1][2][4].
CITATIONS: [(1, '677db8f496af37f7b81c9549262712f6831b507a'), (2, 'd81269cb2d4eb7e0c420e02b806f4207225f131b'), (4, 'b83dfba473986699720d9d9658f4b6a77b5a53e9')]
RETRIEVED: ['677db8f496af37f7b81c9549262712f6831b507a', 'd81269cb2d4eb7e0c420e02b806f4207225f131b', 'd587d18b5bbe6edec12ba28fab028e8feacebcbf', 'b83dfba473986699720d9d9658f4b6a77b5a53e9', 'aa4d08adf039cc660da90d35e0e3bb9c476

=== q3 [answerable] Which topics have the biggest gap between external demand and our internal coverage?
ANSWER   : The topics with the biggest gap between external demand and internal coverage are:

1. **Intel and AMD Processor Developments**: External share 5.06%, internal share 0.57%, gap 4.49%.
2. **Oracle Corp. Takeover Bid**: External share 3.91%, internal share 0.02%, gap 3.89%.
3. **Oracle Enterprise Software Innovations**: External share 3.53%, internal share 0.15%, gap 3.38%.
4. **Microsoft Windows Server Release 2006**: External share 4.57%, internal share 1.40%, gap 3.17%.
5. **Windows XP Security Updates**: External share 2.98%, internal share 0.13%, gap 2.84%.

These topics are currently under-covered relative to their external demand.
CITATIONS: []
RETRIEVED: []



=== q4 [answerable] What software did developers build to let the iPhone work on any phone network?
ANSWER   : Developers created a program that allowed iPhone owners to use their devices on various cell phone networks, not just AT&T. This software was demonstrated by a consultant who showed that an iPhone could work on the T-Mobile network after replacing the AT&T SIM card with a T-Mobile SIM card. The process was described as a "completely software hacked" solution, requiring no physical alterations to the device [3].
CITATIONS: [(3, 'f0d93bdd383762936746b48f7e67ffe72cfb1ede')]
RETRIEVED: ['b6d2522bcf227081a406f782664fc4014607e76f', '630aeccdb47abaa617fa8da1acf0babc4b2b5cd8', 'f0d93bdd383762936746b48f7e67ffe72cfb1ede', 'e0225bd49ed573b78cf3b58d8a6c07ecce2c92cd', '76cf9f58a58a7f0ffe0a55a170b2727e115c7286']



=== q5 [answerable] What incident triggered the classified cyberattack described by Deputy Defense Secretary William Lynn?
ANSWER   : The incident that triggered the classified cyberattack described by Deputy Defense Secretary William Lynn involved an infected laptop at a U.S. base in the Middle East. A malicious flash drive, placed there by a foreign intelligence agency, uploaded its code onto a network operated by the U.S. Central Command. This code spread undetected across both classified and unclassified systems, creating a digital beachhead for data transfer to foreign-controlled servers. Lynn characterized this as a significant breach, highlighting the vulnerability of U.S. military networks to such cyber intrusions [2].
CITATIONS: [(2, '17662736c0b0056d18a33c078f04b64bc369ab0b')]
RETRIEVED: ['4111e42b17c8dfcb6caf9640d1be216a954056b5', '17662736c0b0056d18a33c078f04b64bc369ab0b', '51ed508256b434434c7a0c43741e01f979bdf6f4', 'f5201d3d1b8d44be0d6ae54a4121ad1271b92722', '5fdae928d0419

=== q6 [answerable] What is cyclist Floyd Landis accused of hacking?
ANSWER   : Floyd Landis is accused of hacking into the computer system of the French anti-doping agency's laboratory. A French judge issued an arrest warrant for him on January 28, related to the alleged hacking that took place in 2006. This incident occurred after Landis was stripped of his 2006 Tour de France title due to a positive test for synthetic testosterone. The warrant's issuance came more than three years after the alleged crime, and the reasons for the delay were not immediately clear [1].
CITATIONS: [(1, '6215e369628ef593c7022b497ea4f219f40aad0a')]
RETRIEVED: ['6215e369628ef593c7022b497ea4f219f40aad0a', 'c979e9cc04d104a445e96caf4bf80e29b4fece97', '1218935ba6042ac4af6cb3dfd4bd6fdd685a05e6', '5717dfb586832f1fcc284b5c8b3de06c7f148ef5', '81f6c628b3492e4cb36a63d402f0776869920fb3']



=== q7 [answerable] What does the "How old do I look" website do?
ANSWER   : The "How old do I look" website, developed by Microsoft, analyzes images to estimate the age of the person in the photo. Users can upload their own images or select images found on Bing. The site also allows users to search for celebrities and see how accurately Microsoft can guess their ages. For example, it estimated that 69-year-old Helen Mirren looked 55, while it accurately guessed Kim Kardashian's age as 34, but thought Kanye West, who is 37, looked 38 [1].
CITATIONS: [(1, '4eed9e12497734b4420aebd1ca8351caa096ae1b')]
RETRIEVED: ['4eed9e12497734b4420aebd1ca8351caa096ae1b', '12c3ef272dcf04d11e213dda07ec2793ad70343a', 'df2a1d0d353eb996e65f91197059dfd561fbe771', 'a8d225e1d4042e5c2a6052c00f6432dbe5eb9755', 'b90ed6756be9fc202bcb5775b0606be77699d36d']



=== q8 [answerable] What does the "Anatomy Lab" iPhone app let users do?
ANSWER   : The "Anatomy Lab" iPhone app allows users to perform a virtual dissection of a human body by navigating through 40 body layers. Developed by researchers from the University of Utah, it provides images of a real human cadaver, making it particularly useful for medical and anatomy students who may not have access to actual dissection opportunities. Additionally, medical professionals, especially in fields like physical therapy, use the app to educate patients about body structures. Users can zoom in on different structures and rotate them for various viewpoints, enhancing their understanding of human anatomy.
CITATIONS: []
RETRIEVED: ['b69eedb05cf2556b1a450c6d017fcf1fff26ec97', '441276e12148aa9c70c501330412c165b047ede4', 'b335e7c84252e83715be05ac830c287dda513ae1', 'c8c97eaa5b9a493c6ae98db0909ff287cbe77b7c', '1215571c2382548dafaa1224be600a6556ef048a']



=== q9 [unanswerable] What did NASA announce about its 2024 Artemis III moon landing crew?
ANSWER   : Not found in the library.
CITATIONS: []
RETRIEVED: ['45966f39313c615b0e0ec088a523dcdfd29bf483', '7a56ceed43af91e2877531e188e03c42e4126173', 'ea6d4fb5f45851d780d350eddf69d38f3415a6d8', '016384116ef226ef6dafdad23e395f78608eb758', 'd6e2c349cd63cea90bde50e0b3a2e6d59bbab816', 'ea6d4fb5f45851d780d350eddf69d38f3415a6d8', '7a56ceed43af91e2877531e188e03c42e4126173', '016384116ef226ef6dafdad23e395f78608eb758', '19971969bbb6daee50d6b80755855d5aa1e1fe34', '483f14fcd57719cbbfb58c62a4b259238d064bec', 'ea6d4fb5f45851d780d350eddf69d38f3415a6d8', '7a56ceed43af91e2877531e188e03c42e4126173', '19971969bbb6daee50d6b80755855d5aa1e1fe34', '45966f39313c615b0e0ec088a523dcdfd29bf483', '118b287007d75ab8c9adb10c2556da6ee7685d6f', 'ea6d4fb5f45851d780d350eddf69d38f3415a6d8', '705b09a89de73ae5867d588f4612a82fb1334e0f', '7a56ceed43af91e2877531e188e03c42e4126173', '6db519f381fee3a37504b020cbb9197f56ed0ca8', 'afa33d563

=== q10 [unanswerable] How much did OpenAI raise in its most recent funding round in 2025?
ANSWER   : Not found in the library.
CITATIONS: []
RETRIEVED: ['73413ebbbca088047dd9f8e986affed1b684538a', '4160464d3fcba547cb1664e54ce51ad06f78b773', '905d61704d1206320da1f5c3b72ba280ea0817d7', 'c6060771ec8000cfe41238f6b9f9268085b433c6', 'b04702f0c7a1448724d4a4056dc409b05e5cb01b']

Saved /Users/shubhamyadav/Desktop/topic_modelling/case-study/outputs/rag_results.csv -> fill citation_correct / hallucination with Y/N, then save as eval/rag_results_labeled.csv


### Error analysis

Checked all 10 answers by hand against the source articles and the gap table.

Retrieval: 7 of 7 fact/story questions found the right article in the top 5.

Citations: 6 of 7 correct (86%). The one miss (q8) was a fully accurate answer that just didn't
include the `[n]` marker - a citation omission, not a wrong citation.

Hallucination: 0 of 10. Every answer traces back to real text in its retrieved source, with no
distortion of fact or certainty.

q3 (gap/strategy question): correct, ranked by the same share-based `gap` metric notebook 02 uses.

Both unanswerable questions were correctly declined.

Known scope limit (7.5): gap/coverage questions only cover the 32 topics already identified - a
topic outside that list isn't supported yet, by design.

**Note on run-to-run variation:** these results are from a single run. In an earlier run, q2
overstated the source's certainty ("claimed to have accessed" vs. "tried to access"); the final run
did not. With only 10 questions and some run-to-run wording variation, read 0% hallucination as
"none observed in this run," not as a guarantee.

## Final numbers

Run after filling `citation_correct` and `hallucination` (Y/N) in `outputs/rag_results.csv` and saved it as `eval/rag_results_labeled.csv`.

In [15]:
labeled_path = EVAL_DIR / "rag_results_labeled.csv"
if not labeled_path.exists():
    print("Waiting for eval/rag_results_labeled.csv (human labels)")
else:
    lab = pd.read_csv(labeled_path, dtype=str, keep_default_na=False)

    def yes_rate(col):
        v = lab[col].str.strip().str.upper()
        v = v[v.isin(["Y", "N"])]
        return 100 * (v == "Y").mean() if len(v) else float("nan")

    answerable = lab[lab["type"] == "answerable"]
    # Drop rows with no hit_at_5 value (the gap/strategy question isn't retrieval-based, so it was
    # never checked against a gold_doc_id) - otherwise it would count as a denominator miss below.
    hits = pd.to_numeric(answerable["hit_at_5"], errors="coerce").dropna()
    unans = lab[lab["type"] == "unanswerable"]
    declined = unans["answer"].str.strip().str.rstrip(".").str.lower() == NOT_FOUND.rstrip(".").lower()

    print(f"Hit@5: {int(hits.sum())} of {len(hits)}")
    print(f"Citation accuracy: {yes_rate('citation_correct'):.0f}%")
    print(f"Hallucination rate: {yes_rate('hallucination'):.0f}%")
    print(f"Unanswerable declined: {int(declined.sum())} of {len(unans)}")

Hit@5: 7 of 7
Citation accuracy: 86%
Hallucination rate: 0%
Unanswerable declined: 2 of 2


### `outputs/rag_demo.md` - all 10 questions

Runs after the labels above so each question can show its actual eval result (retrieval hit,
citation correctness, hallucination check) next to the answer, plus a summary table at the end. If
`eval/rag_results_labeled.csv` doesn't exist yet, this still writes all 10 Q&A pairs, just without
the eval line and summary table.

In [16]:
lab_map = {}
if labeled_path.exists():
    lab_map = pd.read_csv(labeled_path, dtype=str, keep_default_na=False).set_index("qid").to_dict("index")

def _yn(value, is_hit=False):
    if is_hit:
        return {"1": "Y", "0": "N"}.get(value, "N/A")
    return value if value in ("Y", "N") else "N/A"

lines = ["# RAG Demo\n"]
for row in test_questions.itertuples(index=False):
    qid = row.qid
    if qid not in all_asks:
        continue
    result = all_asks[qid]

    lines.append(f"## {qid}: {row.question}\n")
    lines.append(f"**Answer:** {result['answer']}\n")
    if result["citations"]:
        lines.append("**Citations:**\n")
        for c in result["citations"]:
            lines.append(f"- [{c['n']}] {c['doc_label']} (doc_id={c['doc_id']}, chunk={c['chunk_id']}): "
                         f"\"{c['snippet']}\"")
        lines.append("")

    labels = lab_map.get(qid, {})
    hit = _yn(labels.get("hit_at_5", ""), is_hit=True)
    cite = _yn(labels.get("citation_correct", ""))
    hall = _yn(labels.get("hallucination", ""))
    lines.append(f"*Correct source retrieved: {hit} | Citation correct: {cite} | Hallucination: {hall}*\n")

if labeled_path.exists():
    lines.append("## Summary\n")
    lines.append("| Metric | Result |")
    lines.append("|---|---|")
    lines.append(f"| Hit@5 | {int(hits.sum())} of {len(hits)} |")
    lines.append(f"| Citation accuracy | {yes_rate('citation_correct'):.0f}% |")
    lines.append(f"| Hallucination rate | {yes_rate('hallucination'):.0f}% |")
    lines.append(f"| Unanswerable declined | {int(declined.sum())} of {len(unans)} |")

with open(OUTPUTS_DIR / "rag_demo.md", "w") as f:
    f.write("\n".join(lines))
print(f"Saved {OUTPUTS_DIR / 'rag_demo.md'}")

Saved /Users/shubhamyadav/Desktop/topic_modelling/case-study/outputs/rag_demo.md


## Try it live

Launches the Streamlit test UI (`app.py`) in the background, reusing everything already built above
(chunks, embeddings, FAISS index, gap DB). No rebuild. Running this notebook end to end gets you a
working link at the end instead of having to separately run `streamlit run app.py` yourself.

In [17]:
import subprocess
import socket
import sys
import time

STREAMLIT_PORT = 8501

def _port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0

if _port_open(STREAMLIT_PORT):
    print(f"Streamlit app already running: http://localhost:{STREAMLIT_PORT}")
else:
    # start_new_session detaches the process from this kernel so it survives after the kernel
    # shuts down (otherwise it would die along with it).
    subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", str(ROOT / "app.py"),
         "--server.headless", "true", "--server.port", str(STREAMLIT_PORT)],
        cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True,
    )
    for _ in range(20):
        if _port_open(STREAMLIT_PORT):
            break
        time.sleep(0.5)
    print(f"Streamlit app running: http://localhost:{STREAMLIT_PORT}")

Streamlit app running: http://localhost:8501
